# Binary vs. Multi-class Decision

## 1. Configuration

In [ ]:
from pathlib import Path
import numpy as np
import torch

PROJECT_ROOT = Path(r"D:\Ravishi\MSc Final Project\skin-lesion-xai")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "decision"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_TRAIN = DATA_PROCESSED / "train.csv"
SPLIT_VAL = DATA_PROCESSED / "val.csv"

IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
RANDOM_SEED = 42
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DX_ORDER = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
DX_TO_IDX = {c: i for i, c in enumerate(DX_ORDER)}

def set_seed(seed=RANDOM_SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cpu":
    print("WARNING: running on CPU")

## 2. Dataset

In [ ]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class DecisionDataset(Dataset):
    def __init__(self, csv_path, mode: str, train: bool):
        self.df = pd.read_csv(csv_path)
        self.mode = mode
        if "dx" not in self.df.columns:
            raise ValueError(f"{csv_path} has no 'dx' column")
        mean, std = IMAGENET_MEAN, IMAGENET_STD
        if train:
            self.tf = transforms.Compose([
                transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0)),
                transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                transforms.RandomRotation(30),
                transforms.ToTensor(), transforms.Normalize(mean, std),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.ToTensor(), transforms.Normalize(mean, std),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.tf(Image.open(row["image_path"]).convert("RGB"))
        dx = str(row["dx"]).strip().lower()
        label = (1 if dx == "mel" else 0) if self.mode == "binary" else DX_TO_IDX[dx]
        return img, torch.tensor(label, dtype=torch.long)

    @property
    def dx_labels(self):
        return self.df["dx"].str.strip().str.lower().values

print("Dataset class ready.")


## 3. Training + evaluation function

In [ ]:
import timm
import torch.nn as nn
import numpy as np
import time
from torch.utils.data import DataLoader

def class_weights_for(dx_labels, mode):
    if mode == "binary":
        y = np.array([1 if d == "mel" else 0 for d in dx_labels]); k = 2
    else:
        y = np.array([DX_TO_IDX[d] for d in dx_labels]); k = len(DX_ORDER)
    counts = np.bincount(y, minlength=k).astype(float)
    counts[counts == 0] = 1.0
    return torch.tensor(len(y) / (k * counts), dtype=torch.float).to(device)


def train_and_eval(mode, epochs):
    print(f"\n{'='*60}\nTraining {mode.upper()} model\n{'='*60}")
    set_seed() 
    n_classes = 2 if mode == "binary" else len(DX_ORDER)

    g = torch.Generator()
    g.manual_seed(RANDOM_SEED)

    train_ds = DecisionDataset(SPLIT_TRAIN, mode, train=True)
    val_ds = DecisionDataset(SPLIT_VAL, mode, train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, generator=g)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS)

    model = timm.create_model("resnet50", pretrained=True, num_classes=n_classes).to(device)
    weights = class_weights_for(train_ds.dx_labels, mode)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time(); running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward(); optimizer.step()
            running += loss.item() * labels.size(0)
        print(f"  epoch {epoch}/{epochs}  loss={running/len(train_ds):.4f}  ({time.time()-t0:.0f}s)")

    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            preds = model(images.to(device)).argmax(1).cpu().numpy()
            all_pred.append(preds); all_true.append(labels.numpy())
    return np.concatenate(all_true), np.concatenate(all_pred)

print("Training function ready.")


## 4. Run both models

In [ ]:
y_true_bin, y_pred_bin = train_and_eval("binary", EPOCHS)
y_true_mc, y_pred_mc = train_and_eval("multiclass", EPOCHS)
print("\nBoth models trained.")


## 5. Confusion matrices 

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

def plot_confusion(y_true, y_pred, labels, names, title, ax):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(names))); ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right"); ax.set_yticklabels(names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    thresh = cm.max() / 2 if cm.max() else 0
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    return cm

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
cm_bin = plot_confusion(y_true_bin, y_pred_bin, [0, 1], ["non-melanoma", "melanoma"],
                        "Binary confusion matrix (validation)", axes[0])
cm_mc = plot_confusion(y_true_mc, y_pred_mc, list(range(len(DX_ORDER))), DX_ORDER,
                       "Multiclass confusion matrix (validation)", axes[1])
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrices_side_by_side.png", dpi=150)
plt.show()

## 6. Per-class precision / recall / F1

In [ ]:
report_bin = classification_report(y_true_bin, y_pred_bin, labels=[0, 1],
                                    target_names=["non-melanoma", "melanoma"],
                                    output_dict=True, zero_division=0)
report_mc = classification_report(y_true_mc, y_pred_mc, labels=list(range(len(DX_ORDER))),
                                   target_names=DX_ORDER, output_dict=True, zero_division=0)

pd.DataFrame(report_bin).transpose().to_csv(RESULTS_DIR / "binary_classification_report.csv")
pd.DataFrame(report_mc).transpose().to_csv(RESULTS_DIR / "multiclass_classification_report.csv")

print(f"BINARY — overall accuracy: {report_bin['accuracy']:.3f}")
for name in ["non-melanoma", "melanoma"]:
    r = report_bin[name]
    print(f"  {name:<14} precision={r['precision']:.2f}  recall={r['recall']:.2f}  "
          f"f1={r['f1-score']:.2f}  (n={int(r['support'])})")

print(f"\nMULTICLASS — overall accuracy: {report_mc['accuracy']:.3f}")
for name in DX_ORDER:
    r = report_mc[name]
    print(f"  {name:<14} precision={r['precision']:.2f}  recall={r['recall']:.2f}  "
          f"f1={r['f1-score']:.2f}  (n={int(r['support'])})")

print("\nKEY OBSERVATION:")
print(f"  Melanoma recall — binary framing:      {report_bin['melanoma']['recall']:.2f}")
print(f"  Melanoma recall — multi-class framing: {report_mc['mel']['recall']:.2f}")